# Global Trends

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [15]:
ROOT_DIR = Path.cwd().parent
Tables_DIR = ROOT_DIR / "data/tables"

In [16]:
monthlyDf = pd.read_csv(Tables_DIR / "table1.csv")
yearlyDf = pd.read_csv(Tables_DIR / "table2.csv")

In [17]:

a1 = yearlyDf.dropna(subset=["co2_mlo_yoy_change"])
g1 = a1[["year", "co2_mlo_yoy_change"]].rename(columns={"co2_mlo_yoy_change": "co2_ppm"})

### How has CO2 changed from 1960 to now?

In [18]:
# 1. Rebuild CO2 level: base (1960) + running total of yearly growth
BASE_1960 = 316.91   # ppm, NOAA Mauna Loa annual mean for 1960
g1 = g1.sort_values("year").reset_index(drop=True).copy()
g1["co2_level"] = BASE_1960 + g1["co2_ppm"].cumsum() - g1["co2_ppm"].iloc[0]

first, last = g1.iloc[0], g1.iloc[-1]
total_rise = last["co2_level"] - first["co2_level"]
pct_rise = total_rise / first["co2_level"] * 100

# 2. Main line
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=g1["year"], y=g1["co2_level"], mode="lines", name="CO2 level",
    line=dict(color="#4C78A8", width=3),
    fill="tozeroy", fillcolor="rgba(76,120,168,0.10)",
    hovertemplate="<b>%{x}</b><br>%{y:.1f} ppm<extra></extra>",
))

# 3. Mark milestones (first year the level passes 350 and 400 ppm)
for m in [350, 400]:
    row = g1[g1["co2_level"] >= m].iloc[0]
    fig.add_trace(go.Scatter(
        x=[row["year"]], y=[row["co2_level"]], mode="markers+text",
        name=f"Passed {m} ppm",
        text=[f"{int(row['year'])}<br>{m} ppm"], textposition="top left",
        marker=dict(size=12, color="#E45756", line=dict(color="white", width=1)),
    ))

# 4. Start and end labels
for r, pos in [(first, "top center"), (last, "top center")]:
    fig.add_trace(go.Scatter(
        x=[r["year"]], y=[r["co2_level"]], mode="markers+text",
        text=[f"{r['co2_level']:.0f} ppm"], textposition=pos,
        marker=dict(size=9, color="#333"), showlegend=False,
    ))

# 5. Styling
fig.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Estimated Atmospheric CO2 Level, 1960-2025</b>"
             f"<br><sup>Up {total_rise:.0f} ppm ({pct_rise:.0f}%) since 1960</sup>",
        x=0.02),
    xaxis_title="Year", yaxis_title="CO2 concentration (ppm)",
    yaxis=dict(range=[290, 450]),
    legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
    margin=dict(t=90, b=100),
    hovermode="x unified",
)
fig.show()

# 6. Summary
print(f"{int(first['year'])}: {first['co2_level']:.1f} ppm")
print(f"{int(last['year'])}:  {last['co2_level']:.1f} ppm")
print(f"Total rise: {total_rise:.1f} ppm ({pct_rise:.0f}% increase)")

1960: 316.9 ppm
2025:  427.4 ppm
Total rise: 110.4 ppm (35% increase)


> Atmospheric CO₂ rose from about **316.9 ppm** in **1960** to about **427.4 ppm** in **2025**, an increase of roughly **110.4 ppm** (about **35%**). The rise has been continuous, and no year shows a decline.

### Is the growth rate of CO2 itself accelerating?

In [19]:
from scipy.stats import linregress

# 1. Statistics
g1 = g1.sort_values("year").reset_index(drop=True).copy()
r = linregress(g1["year"], g1["co2_ppm"])
g1["trend"] = r.intercept + r.slope * g1["year"]
g1["roll5"] = g1["co2_ppm"].rolling(5, center=True, min_periods=3).mean()

# 2. Chart
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=g1["year"], y=g1["co2_ppm"], mode="lines+markers",
    name="Yearly growth", line=dict(color="#9DB7D5", width=1.5),
    marker=dict(size=5),
    hovertemplate="<b>%{x}</b><br>%{y:.2f} ppm/yr<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=g1["year"], y=g1["roll5"], mode="lines", name="5-year rolling average",
    line=dict(color="#F58518", width=3),
))
fig.add_trace(go.Scatter(
    x=g1["year"], y=g1["trend"], mode="lines",
    name=f"Trend (+{r.slope * 10:.2f} ppm/yr per decade)",
    line=dict(color="#E45756", width=2.5, dash="dash"),
))

fig.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Is the CO2 Growth Rate Accelerating?</b>"
             f"<br><sup>Trend slope: {r.slope:.3f} ppm/yr each year "
             f"(p = {r.pvalue:.1e}, R² = {r.rvalue**2:.2f})</sup>",
        x=0.02),
    xaxis_title="Year", yaxis_title="CO2 growth (ppm per year)",
    yaxis=dict(range=[0, 4]),
    legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
    margin=dict(t=90, b=100),
    hovermode="x unified",
)
fig.show()

# 3. Decade averages (less noisy than single years)
dec = g1.groupby(g1["year"] // 10 * 10)["co2_ppm"].mean().round(2)
print("Average growth per decade (ppm/yr):")
print(dec.to_string())
print(f"\nSlope: {r.slope:.4f} ppm/yr per year")
print(f"Growth rate change over the period: {r.slope * (g1['year'].max() - g1['year'].min()):.2f} ppm/yr")

Average growth per decade (ppm/yr):
year
1960    0.86
1970    1.22
1980    1.64
1990    1.53
2000    1.91
2010    2.40
2020    2.62

Slope: 0.0291 ppm/yr per year
Growth rate change over the period: 1.89 ppm/yr


> **Yes**, The annual growth rate of atmospheric CO₂ has accelerated. It increased from an average of **0.86 ppm/yr** in the **1960s** to **2.62 ppm/yr** in the **2020s**, roughly a threefold rise. A linear trend shows the growth rate increasing by about **0.03 ppm/yr** each year (about **0.3 ppm/yr** per decade). The only slowdown, in the **1990s**, coincides with the Mount Pinatubo eruption, and short-term spikes and dips are mainly driven by El Niño and volcanic events on top of this long-term acceleration.

### What is the seasonal cycle of CO2 and which month peaks?

In [20]:
m = monthlyDf.sort_values(["year", "month"]).copy()

r = m["co2_ppm"].rolling(12, center=True).mean()
m["trend"] = (r + r.shift(-1)) / 2          # properly centered 12-month trend

# 2. Seasonal part = CO2 minus the trend (do NOT overwrite this later)
m["anomaly"] = m["co2_ppm"] - m["trend"]
m = m.dropna(subset=["anomaly"])

# 3. Keep only years that still have all 12 months
cnt = m.groupby("year")["month"].nunique()
m = m[m["year"].isin(cnt[cnt == 12].index)].copy()

# 4. Central value per month across ALL years (median)
names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
seasonal = m.groupby("month")["anomaly"].agg(["median", "mean", "std", "count"]).reset_index()
seasonal["name"] = seasonal["month"].apply(lambda i: names[i - 1])

peak = seasonal.loc[seasonal["median"].idxmax()]
trough = seasonal.loc[seasonal["median"].idxmin()]
amplitude = peak["median"] - trough["median"]
n_years = m["year"].nunique()

# 5. Box plot: every year's value for each month
fig = go.Figure()
for i, name in enumerate(names, start=1):
    vals = m.loc[m["month"] == i, "anomaly"]
    fig.add_trace(go.Box(
        y=vals, x=[name] * len(vals), name=name,
        boxpoints="all", jitter=0.4, pointpos=0,
        marker=dict(size=4, opacity=0.5),
        line=dict(width=1.5),
        fillcolor="rgba(228,87,86,0.25)" if vals.median() >= 0 else "rgba(76,120,168,0.25)",
        marker_color="#E45756" if vals.median() >= 0 else "#4C78A8",
        showlegend=False,
        hovertemplate="%{y:+.2f} ppm<extra>" + name + "</extra>",
    ))

# 6. Median curve connecting the 12 months
fig.add_trace(go.Scatter(
    x=seasonal["name"], y=seasonal["median"], mode="lines+markers",
    line=dict(color="#222", width=3), marker=dict(size=8),
    name="Median (central value)",
))

# 7. Peak and lowest month
fig.add_trace(go.Scatter(
    x=[peak["name"]], y=[peak["median"]], mode="markers+text",
    marker=dict(color="red", size=15, line=dict(color="white", width=1.5)),
    text=[f"Peak: {peak['name']}<br>{peak['median']:+.2f}"], textposition="top center",
    name="Peak month",
))
fig.add_trace(go.Scatter(
    x=[trough["name"]], y=[trough["median"]], mode="markers+text",
    marker=dict(color="green", size=15, line=dict(color="white", width=1.5)),
    text=[f"Low: {trough['name']}<br>{trough['median']:+.2f}"], textposition="bottom center",
    name="Lowest month",
))

# 8. Styling
fig.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Seasonal Cycle of CO2</b>"
             f"<br><sup>{n_years} years | Peak in {peak['name']}, lowest in {trough['name']}, "
             f"Swing of {amplitude:.1f} ppm</sup>",
        x=0.02),
    xaxis=dict(title="Month", categoryorder="array", categoryarray=names),
    yaxis=dict(title="CO2 vs 12-month trend (ppm)"),
    legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
    margin=dict(t=90, b=100),
)
fig.add_hline(y=0, line_color="gray", line_dash="dash")
fig.show()

# 9. Printed summary
out = seasonal[["name", "median", "mean", "std", "count"]].round(2)
out.columns = ["Month", "Median", "Mean", "Std", "Years"]
print(out.to_string(index=False))
print(f"\nPeak month:   {peak['name']} ({peak['median']:+.2f} ppm)")
print(f"Lowest month: {trough['name']} ({trough['median']:+.2f} ppm)")
print(f"Seasonal swing: {amplitude:.2f} ppm")

Month  Median  Mean  Std  Years
  Jan    0.96  0.93 0.20     46
  Feb    1.27  1.25 0.21     46
  Mar    1.54  1.52 0.19     46
  Apr    1.80  1.76 0.15     46
  May    1.61  1.61 0.11     46
  Jun    0.59  0.61 0.19     46
  Jul   -1.16 -1.16 0.29     46
  Aug   -2.62 -2.61 0.31     46
  Sep   -2.65 -2.66 0.22     46
  Oct   -1.48 -1.47 0.15     46
  Nov   -0.27 -0.26 0.16     46
  Dec    0.48  0.48 0.19     46

Peak month:   Apr (+1.80 ppm)
Lowest month: Sep (-2.65 ppm)
Seasonal swing: 4.45 ppm


> Atmospheric CO₂ follows a regular seasonal cycle driven. After removing the long-term trend with a 12-month moving average, the median across 46 complete years shows **CO₂ peaking in April (+1.80 ppm above trend)**, with **May close behind (+1.61 ppm)**. It then falls sharply through summer, reaching its **lowest point in September (-2.65 ppm), with August almost equal (-2.62 ppm).** The **seasonal swing is about 4.45 ppm from peak to trough**. The pattern is highly consistent. The cycle arises because plants absorb CO₂ during the summer growing season and release it through decay in autumn and winter. This seasonal swing of about 4.5 ppm is small compared with the long-term rise of roughly 108 ppm since 1960.

### How have CH4 and N2O changed, and are they growing faster or slower than CO2 in relative terms?

In [21]:
# 1. Keep years where all three gases have data
gases = ["co2_ppm", "ch4_ppb", "n2o_ppb"]

d = (
    yearlyDf.dropna(subset=gases)
    .sort_values("year")
    .reset_index(drop=True)
    .copy()
)

if len(d) < 2:
    raise ValueError("At least two complete years are required.")

start = int(d["year"].iloc[0])
end = int(d["year"].iloc[-1])
years = end - start

labels = {
    "co2_ppm": "CO₂",
    "ch4_ppb": "CH₄",
    "n2o_ppb": "N₂O",
}

colors = {
    "co2_ppm": "#3B82F6",
    "ch4_ppb": "#EF4444",
    "n2o_ppb": "#10B981",
}

# 2. Calculate relative changes from the common baseline
for g in gases:
    d[g + "_pct"] = (d[g] / d[g].iloc[0] - 1) * 100

# 3. Create a two-panel figure
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        "Relative growth since the baseline year",
        "Annual change in relative growth (percentage points)",
    ),
)

for g in gases:
    pct_col = g + "_pct"

    # Main cumulative percentage growth line
    fig.add_trace(
        go.Scatter(
            x=d["year"],
            y=d[pct_col],
            mode="lines",
            name=labels[g],
            line=dict(color=colors[g], width=3),
            hovertemplate=(
                "<b>%{x}</b><br>"
                + labels[g]
                + ": %{y:.2f}%<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

    # Endpoint marker and label
    final_pct = d[pct_col].iloc[-1]

    fig.add_trace(
        go.Scatter(
            x=[end],
            y=[final_pct],
            mode="markers+text",
            text=[f"{final_pct:+.1f}%"],
            textposition="middle right",
            textfont=dict(color=colors[g], size=12),
            marker=dict(color=colors[g], size=8),
            showlegend=False,
            hoverinfo="skip",
            cliponaxis=False,
        ),
        row=1,
        col=1,
    )

    # Year-on-year change in the percentage-growth index
    annual_change = d[pct_col].diff()

    fig.add_trace(
        go.Scatter(
            x=d["year"],
            y=annual_change,
            mode="lines",
            name=labels[g] + " annual change",
            line=dict(color=colors[g], width=1.8),
            showlegend=False,
            hovertemplate=(
                "<b>%{x}</b><br>"
                + labels[g]
                + ": %{y:+.3f} percentage points"
                + "<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

fig.update_layout(
    template="plotly_white",
    title=dict(
        text=(
            f"<b>How Have CO₂, CH₄ and N₂O Changed?</b>"
            f"<br><sup>Relative change from {start} to {end}"
            f" | Common baseline for all gases</sup>"
        ),
        x=0.03,
        xanchor="left",
    ),
    height=780,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.01,
        xanchor="left",
        x=0,
    ),
    margin=dict(l=75, r=110, t=125, b=65),
)

fig.update_yaxes(
    title_text="Change since baseline (%)",
    zeroline=True,
    zerolinecolor="#9CA3AF",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Percentage points per year",
    zeroline=True,
    zerolinecolor="#9CA3AF",
    row=2,
    col=1,
)

fig.update_xaxes(title_text="Year", row=2, col=1)
fig.show()

# 4. Calculate statistics for the research question
rows = []

for g in gases:
    initial = d[g].iloc[0]
    final = d[g].iloc[-1]
    total_pct = (final / initial - 1) * 100

    # Geometric annualized growth rate
    annualized_pct = ((final / initial) ** (1 / years) - 1) * 100

    # Absolute concentration change per year
    absolute_per_year = (final - initial) / years

    rows.append({
        "Gas": labels[g],
        "Start concentration": initial,
        "End concentration": final,
        "Absolute change": final - initial,
        "Total change (%)": total_pct,
        "Annualized growth (%)": annualized_pct,
        "Absolute change/year": absolute_per_year,
    })

summary = pd.DataFrame(rows).set_index("Gas")

print(f"Period: {start}–{end} ({years} elapsed years)")
print("\nConcentration and relative growth:")
print(summary.round(3).to_string())

# 5. Compare relative growth against CO2
co2_rate = summary.loc["CO₂", "Annualized growth (%)"]

print()
print("RESEARCH QUESTION: CH4 AND N2O VS CO2")

for gas in ["CH₄", "N₂O"]:
    rate = summary.loc[gas, "Annualized growth (%)"]
    difference = rate - co2_rate

    if difference > 0.001:
        comparison = "faster"
    elif difference < -0.001:
        comparison = "slower"
    else:
        comparison = "at approximately the same rate"

    print(
        f"{gas}: increased {summary.loc[gas, 'Total change (%)']:.2f}% "
        f"overall, with an annualized growth rate of {rate:.3f}%."
    )
    print(
        f"  Relative to CO₂ ({co2_rate:.3f}% annually), "
        f"{gas} grew {comparison} by {abs(difference):.3f} "
        "percentage points per year."
    )



Period: 2001–2025 (24 elapsed years)

Concentration and relative growth:
     Start concentration  End concentration  Absolute change  Total change (%)  Annualized growth (%)  Absolute change/year
Gas                                                                                                                        
CO₂              370.566            425.622           55.057            14.857                  0.579                 2.294
CH₄             1771.320           1935.990          164.670             9.296                  0.371                 6.861
N₂O              316.364            338.864           22.500             7.112                  0.287                 0.938

RESEARCH QUESTION: CH4 AND N2O VS CO2
CH₄: increased 9.30% overall, with an annualized growth rate of 0.371%.
  Relative to CO₂ (0.579% annually), CH₄ grew slower by 0.208 percentage points per year.
N₂O: increased 7.11% overall, with an annualized growth rate of 0.287%.
  Relative to CO₂ (0.579% annually

> From **2001 to 2025**, atmospheric concentrations of all three greenhouse gases increased, but **CO₂ exhibited the highest relative growth.** Its concentration **rose by 14.86%, compared with 9.30% for CH₄ and 7.11% for N₂O.** On an annualized basis, **CO₂ grew at 0.579%, exceeding CH₄ (0.371%) and N₂O (0.287%).** This indicates that CH₄ and N₂O accumulated more slowly than CO₂ in percentage terms over the study period.

### How did global temperature anomaly change by decade, and which decade jumped most?

In [40]:


# 1. Prepare temperature data
t1 = (
    yearlyDf[["year", "temp_anomaly"]]
    .dropna()
    .sort_values("year")
    .reset_index(drop=True)
    .copy()
)

t1["year"] = t1["year"].astype(int)
t1["decade"] = (t1["year"] // 10) * 10

# 2. Calculate decade averages and sample counts
decadal = (
    t1.groupby("decade")
    .agg(
        mean_anomaly=("temp_anomaly", "mean"),
        min_anomaly=("temp_anomaly", "min"),
        max_anomaly=("temp_anomaly", "max"),
        years_available=("temp_anomaly", "count"),
    )
    .reset_index()
)

# Keep the decade with the most recent year, even if incomplete
start_year = int(t1["year"].min())
end_year = int(t1["year"].max())

# 3. Calculate linear trend across decade averages
x = decadal["decade"].to_numpy()
y = decadal["mean_anomaly"].to_numpy()

slope, intercept = np.polyfit(x, y, 1)
decadal["trend"] = slope * x + intercept

# 4. Identify latest decade
latest = decadal.iloc[-1]
first = decadal.iloc[0]
change = latest["mean_anomaly"] - first["mean_anomaly"]

# 5. Plot
fig = go.Figure()

# Range of yearly values within each decade
fig.add_trace(
    go.Scatter(
        x=decadal["decade"],
        y=decadal["max_anomaly"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
    )
)

fig.add_trace(
    go.Scatter(
        x=decadal["decade"],
        y=decadal["min_anomaly"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(76, 120, 168, 0.12)",
        line=dict(width=0),
        name="Yearly range within decade",
        hoverinfo="skip",
    )
)

# Decade averages
fig.add_trace(
    go.Scatter(
        x=decadal["decade"],
        y=decadal["mean_anomaly"],
        mode="lines+markers",
        name="Decade average",
        line=dict(color="#4C78A8", width=3),
        marker=dict(size=7),
        customdata=decadal[["years_available"]],
        hovertemplate=(
            "<b>%{x}s</b><br>"
            "Mean anomaly: %{y:+.2f} °C<br>"
            "Years available: %{customdata[0]}"
            "<extra></extra>"
        ),
    )
)

# Linear trend
fig.add_trace(
    go.Scatter(
        x=decadal["decade"],
        y=decadal["trend"],
        mode="lines",
        name="Linear trend",
        line=dict(color="#E45756", width=2, dash="dash"),
        hovertemplate="Trend: %{y:+.2f} °C<extra></extra>",
    )
)

# Zero anomaly reference
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="#777777",
    annotation_text="Zero anomaly",
    annotation_position="bottom right",
)

fig.update_layout(
    template="plotly_white",
    title=dict(
        text=(
            "<b>Long-Term Change in Global Surface Temperature</b>"
            f"<br><sup>Decade averages | {start_year}–{end_year}</sup>"
        ),
        x=0.03,
        xanchor="left",
    ),
    xaxis_title="Decade",
    yaxis_title="Temperature anomaly (°C)",
    hovermode="x unified",
    height=600,
    margin=dict(l=75, r=40, t=110, b=80),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
    ),
)

fig.update_xaxes(dtick=10)
fig.update_yaxes(zeroline=False, ticksuffix="°C")

fig.show()

# 6. Print summary statistics
print(f"Period: {start_year}–{end_year}")
print(f"Decades represented: {len(decadal)}")
print(f"Difference between these decade averages: {change:+.3f} °C")
print(f"Linear trend across decade averages: {slope:+.4f} °C/year")


Period: 1880–2025
Decades represented: 15
Difference between these decade averages: +1.284 °C
Linear trend across decade averages: +0.0087 °C/year


Global temperature anomalies show a clear long-term warming trend, rising from an average of **−0.217°C in the 1880s** to **+1.067°C in the 2020s**, a difference of  **1.284°C between the two decade averages**. After relatively cooler conditions through the early 20th century and modest fluctuations around the mid-century period, warming **accelerated from the 1980s onward**. The largest increase between consecutive decade averages occurred from the **2010s to the 2020s, rising by 0.261°C**. This is notable because the 2020s data currently cover only six years (2020–2025), so the decade is incomplete. Overall, the data indicate that global warming has intensified in recent decades, with the latest observations substantially above the historical baseline.